# QR Phishing Detection — URL Model Training (Colab)

Trains a character-level CNN to detect phishing **URLs**. A QR code is just a container for a
URL, so the phishing signal is in the URL text. The app scans the QR image, decodes it to a URL,
and this model judges the URL.

**Before running:** download `malicious_phish.csv` from
https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset and put it in your Google Drive
at `/content/drive/MyDrive/malicious_phish.csv`. Then: **Runtime → Run all**.

## SECTION 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## SECTION 2: Load the URL dataset

We read the CSV, find the URL and label columns automatically, and convert the label to binary
(0 = benign / safe, 1 = phishing). Other categories (malware/defacement) are dropped so the model
focuses on phishing vs benign.

In [ ]:
import pandas as pd

CSV_PATH = '/content/drive/MyDrive/malicious_phish.csv'  # <-- change if you put it elsewhere

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower() for c in df.columns]
print('Columns:', list(df.columns))

# Locate the URL column and the label column.
url_col = 'url' if 'url' in df.columns else df.columns[0]
label_col = next((c for c in ['type', 'label', 'result', 'class', 'status'] if c in df.columns),
                 df.columns[-1])
print(f'Using url column = "{url_col}", label column = "{label_col}"')


def to_binary(v):
    s = str(v).strip().lower()
    if s in ('benign', 'legitimate', 'legit', 'good', 'safe', '0'):
        return 0
    if s in ('phishing', 'phish', 'malicious', 'bad', '1'):
        return 1
    return None  # drop malware/defacement/unknown


df = df[[url_col, label_col]].dropna()
df['y'] = df[label_col].map(to_binary)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
df = df[df[url_col].astype(str).str.len() > 3]

print('\nClass counts (0=benign, 1=phishing):')
print(df['y'].value_counts())
print('\nSample rows:')
print(df.sample(min(8, len(df)), random_state=1)[[url_col, 'y']].to_string(index=False))

## SECTION 3: Balance the classes and split

We take an equal number of benign and phishing URLs so the model can't cheat on class size,
then split 70 / 15 / 15 into train / validation / test. We keep the raw URL **text**.

In [ ]:
from sklearn.model_selection import train_test_split

SAMPLES_PER_CLASS = 30000  # raise for higher accuracy / lower for speed

benign = df[df['y'] == 0]
phish = df[df['y'] == 1]
n = min(SAMPLES_PER_CLASS, len(benign), len(phish))
print(f'Using {n} per class ({2 * n} total).')

balanced = pd.concat([
    benign.sample(n, random_state=42),
    phish.sample(n, random_state=42),
]).sample(frac=1, random_state=42)  # shuffle

urls = balanced[url_col].astype(str).tolist()
import numpy as np
y_all = balanced['y'].to_numpy()

urls_train, urls_temp, y_train, y_temp = train_test_split(
    urls, y_all, test_size=0.3, random_state=42, stratify=y_all)
urls_val, urls_test, y_val, y_test = train_test_split(
    urls_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f'Train {len(urls_train)} | Val {len(urls_val)} | Test {len(urls_test)}')

## SECTION 4: Normalize URLs + character tokenizer

`normalize_url` strips formatting artifacts (scheme, `www.`, case) so the model learns real
content. **This is identical to the backend's `ml_service.py`** so training and the app match.
Then each URL becomes a fixed-length sequence of character ids (id 0 = pad, id 1 = unknown).

In [ ]:
import numpy as np

MAXLEN = 200


def normalize_url(url):
    u = (url or '').strip().lower()
    if '://' in u:
        u = u.split('://', 1)[1]   # drop http:// / https://
    while u.startswith('www.'):
        u = u[4:]                  # drop leading www.
    return u


# Build vocab from TRAIN urls only (no leakage). chars start at id 2.
chars = sorted(set(''.join(normalize_url(u) for u in urls_train)))
char_index = {c: i + 2 for i, c in enumerate(chars)}
VOCAB_SIZE = len(char_index) + 2
print(f'Vocab size = {VOCAB_SIZE}, maxlen = {MAXLEN}')


def encode(url):
    seq = [char_index.get(ch, 1) for ch in normalize_url(url)[:MAXLEN]]
    return seq + [0] * (MAXLEN - len(seq))


X_train = np.array([encode(u) for u in urls_train], dtype=np.int32)
X_val = np.array([encode(u) for u in urls_val], dtype=np.int32)
X_test = np.array([encode(u) for u in urls_test], dtype=np.int32)
print('Encoded shapes:', X_train.shape, X_val.shape, X_test.shape)

## SECTION 5: Build and train the model (modern Transformer-style)

A character-level network with a **Transformer self-attention** block — the modern standard for
URL/text classification. `Embedding -> Conv1D (local context) -> Multi-Head Self-Attention ->
GlobalMaxPooling -> Dense`. It uses only standard Keras layers, so the backend can load it with
no custom code. Self-attention lets the model weigh suspicious parts of the URL (host, TLD, path
tokens) against each other instead of reading it left-to-right only.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Modern attention-based (Transformer encoder) URL classifier.
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128

inputs = layers.Input(shape=(MAXLEN,))
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(inputs)
x = layers.Conv1D(EMBED_DIM, 5, padding='same', activation='relu')(x)  # adds positional context

# --- Transformer encoder block ---
attn = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=EMBED_DIM)(x, x)
attn = layers.Dropout(0.1)(attn)
x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
ff = layers.Dense(FF_DIM, activation='relu')(x)
ff = layers.Dense(EMBED_DIM)(ff)
ff = layers.Dropout(0.1)(ff)
x = layers.LayerNormalization(epsilon=1e-6)(x + ff)
# ---------------------------------

x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-5)

print('Training...')
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
)
print('Done.')

## SECTION 6: Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {acc:.4f}  |  Test loss: {loss:.4f}')

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int)
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['benign', 'phishing']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['benign', 'phishing'], yticklabels=['benign', 'phishing'])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.show()

## SECTION 7: Generalization test (real-world URLs)

Tests the model on URLs NOT in the dataset — legit sites (with/without `www.`, with paths) and
clear phishing. A good model scores legit LOW and phishing HIGH regardless of `www.`.

In [ ]:
def score(u):
    return float(model.predict(np.array([encode(u)], dtype=np.int32), verbose=0)[0][0])

legit = ['https://www.google.com', 'https://google.com', 'https://github.com/login',
         'https://en.wikipedia.org/wiki/QR_code', 'https://www.amazon.com/gp/cart',
         'https://www.linkedin.com/feed', 'https://youtube.com/watch?v=abc']
phish = ['http://192.168.0.5@paypal-secure.tk/login/verify', 'http://free-gift-card.tk/claim/password',
         'http://45.137.21.9/secure/signin/confirm', 'http://bit.ly/3xPhish',
         'https://www.appleid-verify.tk/login']

print('LEGIT (want LOW):')
okl = 0
for u in legit:
    p = score(u); okl += p < 0.5; print(f'  {p*100:6.1f}%  {u}')
print('PHISHING (want HIGH):')
okp = 0
for u in phish:
    p = score(u); okp += p > 0.5; print(f'  {p*100:6.1f}%  {u}')
print(f'\nGeneralization: legit {okl}/{len(legit)}, phishing {okp}/{len(phish)}')

## SECTION 8: Export model + tokenizer (download these 2 files)

Saves `phishing_url_model.keras` and `url_tokenizer.json` to Drive, then verifies the model
separates the classes. Download BOTH into the backend folder
`qr-code-fishing-backend/app/models/ml/`.

In [ ]:
import os, json
import tensorflow as tf

out_dir = '/content/drive/MyDrive'
model_path = os.path.join(out_dir, 'phishing_url_model.keras')
tok_path = os.path.join(out_dir, 'url_tokenizer.json')

model.save(model_path)
with open(tok_path, 'w', encoding='utf-8') as f:
    json.dump({'char_index': char_index, 'maxlen': MAXLEN}, f)
print('Saved model     ->', model_path)
print('Saved tokenizer ->', tok_path)

m2 = tf.keras.models.load_model(model_path)
ben = X_test[y_test == 0][:1000]
mal = X_test[y_test == 1][:1000]
bs = float(m2.predict(ben, verbose=0).mean())
ms = float(m2.predict(mal, verbose=0).mean())
spread = ms - bs
print(f'Mean P(phishing): benign={bs:.3f}  phishing={ms:.3f}  spread={spread:.3f}')
assert spread > 0.30, f'NOT discriminating (spread={spread:.3f}); train longer / more data.'
print('OK. Download BOTH files into qr-code-fishing-backend/app/models/ml/')